In [1]:
!pip install faker

In [6]:
import csv
import random
import pandas as pd
from faker import Faker

fake = Faker()

num_files = 5
emails_per_file = 10000
labels = ['spam', 'legit', 'phishing']  # tu peux ajouter d'autres catégories

for file_index in range(1, num_files + 1):
    filename = f'emails_{file_index}.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['id', 'sender', 'recipient', 'subject', 'body', 'label'])

        for i in range(1, emails_per_file + 1):
            mail_id = f'{file_index}_{i}'
            sender = fake.email()
            recipient = fake.email()
            subject = fake.sentence(nb_words=random.randint(3, 8))
            body = fake.paragraph(nb_sentences=random.randint(2, 5))
            label = random.choice(labels)
            writer.writerow([mail_id, sender, recipient, subject, body, label])

    print(f'{filename} créé avec succès.')


emails_1.csv créé avec succès.
emails_2.csv créé avec succès.
emails_3.csv créé avec succès.
emails_4.csv créé avec succès.
emails_5.csv créé avec succès.


In [9]:
import pandas as pd


def load_raw_datasets():
        # Charger chaque CSV
    emails_1 = pd.read_csv("emails_1.csv", sep=",", quotechar='"', engine="python")
    emails_2 = pd.read_csv("emails_2.csv", sep=",", quotechar='"', engine="python")
    emails_3 = pd.read_csv("emails_3.csv", sep=",", quotechar='"', engine="python")
    emails_4 = pd.read_csv("emails_4.csv", sep=",", quotechar='"', engine="python")
    emails_5 = pd.read_csv("emails_5.csv", sep=",", quotechar='"', engine="python")

    
    return emails_1, emails_2, emails_3, emails_4, emails_5



emails_1, emails_2, emails_3, emails_4, emails_5 = load_raw_datasets()
   
print("Emails_1:", len(emails_1))
print("Emails_2:", len(emails_2))
print("Emails_3:", len(emails_3))
print("Emails_4:", len(emails_4))
print("Emails_5:", len(emails_5))


Emails_1: 10000
Emails_2: 10000
Emails_3: 10000
Emails_4: 10000
Emails_5: 10000


In [11]:
import pandas as pd


def prepare_dataset():
   
    emails_1, emails_2, emails_3, emails_4, emails_5 = load_raw_datasets()

   
    df = pd.concat([emails_1, emails_2, emails_3, emails_4, emails_5], ignore_index=True)

    df.drop_duplicates(inplace=True)
    df.dropna(subset=["body"], inplace=True)

    return df

if __name__ == "__main__":
    df = prepare_dataset()
    print("Emails par classe:")
    print(df["label"].value_counts())

    df.to_csv("../processed/all_emails.csv", index=False)
    print("Dataset sauvegardé dans ../processed/all_emails.csv")


Emails par classe:
label
phishing    16709
spam        16690
legit       16601
Name: count, dtype: int64
Dataset sauvegardé dans ../processed/all_emails.csv


In [21]:
# def compute_risk_score(text, model, vectorizer):

#     #--- règles sur texte brut ---
#     score_rules, reasons = urgent_words_rules(text)

#     # --- score ML ---
#     text_vec = vectorizer.transform([text])
#     probas = model.predict_proba(text_vec)[0]

#     classes = list(model.classes_)

#     if "phishing" in classes:
#         idx = classes.index("phishing")
#         score_ml = probas[idx] * 100
#     else:
#         score_ml = 0

#     score_final = min(score_ml * 0.7 + score_rules * 0.3, 100)

#     return score_final, reasons

from networkx import attr_matrix
from numpy import hstack

def extract_basic_features(text):
    text_lower = text.lower()
    return {
        "length": len(text),
        "num_words": len(text.split()),
        "num_exclam": text.count("!"),
        "num_links": text_lower.count("http"),
        "num_digits": sum(c.isdigit() for c in text),
    }


from scipy.sparse import hstack, csr_matrix

def compute_risk_score(email, model, vectorizer):
    # TF-IDF
    tfidf_vec = vectorizer.transform([email.lower()])
    # Features basiques
    basic_feat = pd.DataFrame([extract_basic_features(email)])
    # Combiner correctement avec csr_matrix
    combined = hstack([tfidf_vec, csr_matrix(basic_feat.values)])
    
    # Prédiction
    probas = model.predict_proba(combined)[0]
    classes = list(model.classes_)
    
    score = probas[classes.index("phishing")] if "phishing" in classes else max(probas)
    reasons = {
        "length": basic_feat["length"].iloc[0],
        "num_words": basic_feat["num_words"].iloc[0],
        "num_exclam": basic_feat["num_exclam"].iloc[0],
        "num_links": basic_feat["num_links"].iloc[0],
        "num_digits": basic_feat["num_digits"].iloc[0],
    }
    return score, reasons


def urgent_words_rules(text):

    mots_urgents = [
        "urgent", "verify", "now", "click",
        "confirm", "action required",
        "password", "billing", "login",
        "security alert"
    ]

    score_rules = 0
    reasons = []

    text = text.lower()

    for word in mots_urgents:
        if word in text:
            score_rules += 20
            reasons.append(word)

    return score_rules, reasons

In [ ]:


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from scipy.sparse import hstack
import joblib


# ------------------------------
# 1. Charger les données
# ------------------------------
df = pd.read_csv("../processed/all_emails.csv", sep=",", quotechar='"', engine="python", encoding="utf-8")

# ------------------------------
# 2. Préparer le texte
# ------------------------------
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
df['text'] = df['subject'] + ' ' + df['body']

# Nettoyage simple
# Lowercase the text first, then replace non-alphanumeric characters with a space
df['text'] = df['text'].str.lower().str.replace(r'[^a-z0-9\s]', ' ', regex=True)
df = df[df['text'].str.strip() != '']  # supprimer textes vides

# ------------------------------
# 3. Features basiques
# ------------------------------
def extract_basic_features(text):
    text_lower = text.lower()
    return {
        "length": len(text),
        "num_words": len(text.split()),
        "num_exclam": text.count("!"),
        "num_links": text_lower.count("http"),
        "num_digits": sum(c.isdigit() for c in text),
    }

df['spam'] = df['label'].apply(lambda x: 1 if x == 'spam' else 0)
# ------------------------------
# 4. Séparer features et labels
# ------------------------------
X = df['text']
y = df['spam'] # 1 pour spam, 0 pour legit

# ------------------------------
# 5. Split train/test
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------------
# 6. TF-IDF vectorization
# ------------------------------
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ------------------------------
# 7. Extraire features basiques
# ------------------------------
basic_train = pd.DataFrame([extract_basic_features(t) for t in X_train])
basic_test = pd.DataFrame([extract_basic_features(t) for t in X_test])

from scipy.sparse import csr_matrix

# ------------------------------
from scipy.sparse import csr_matrix

X_train_combined = hstack([X_train_tfidf, csr_matrix(basic_train.values)])
X_test_combined = hstack([X_test_tfidf, csr_matrix(basic_test.values)])
basic_train_sparse = csr_matrix(basic_train.values)
basic_test_sparse = csr_matrix(basic_test.values)
X_train_combined = hstack([X_train_tfidf, basic_train_sparse])
X_test_combined = hstack([X_test_tfidf, basic_test_sparse])

# ------------------------------
# 9. Modèle Logistic Regression
# ------------------------------
clf = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',  # multiclass supporté
    random_state=42
)
clf.fit(X_train_combined, y_train)

# ------------------------------
# 10. Évaluation
# ------------------------------
y_pred = clf.predict(X_test_combined)

print("=== Classification Report ===\n")
print(classification_report(y_test, y_pred))

print("=== Matrice de Confusion ===\n")
print(confusion_matrix(y_test, y_pred))

# ------------------------------
# 11. Sauvegarde modèle et vectorizer
# ------------------------------
joblib.dump(clf, "../../models/phishing_model.joblib")
joblib.dump(vectorizer, "../../models/tfidf_vectorizer.joblib")
print("\nModèle et vectorizer sauvegardés dans le dossier 'models/'")

# ------------------------------
# 12. Prédiction individuelle avec compute_risk_score
# ------------------------------
sample_email = "Urgent: verify your account now by clicking this link"

# Créer la combinaison TF-IDF + features basiques pour le sample
sample_vec = vectorizer.transform([sample_email.lower()])
sample_basic = pd.DataFrame([extract_basic_features(sample_email)])
sample_combined = hstack([sample_vec, sample_basic.values])
from scipy.sparse import hstack

def compute_combined_features(email, vectorizer):
    tfidf_vec = vectorizer.transform([email.lower()])
    basic_feat = pd.DataFrame([extract_basic_features(email)])
    return hstack([tfidf_vec, basic_feat.values])

sample_features = compute_combined_features(sample_email, vectorizer)
#score, reasons = compute_risk_score(sample_email, clf, vectorizer)
# Utiliser compute_risk_score pour obtenir le score et les raisons
score, reasons = compute_risk_score(sample_email, clf, vectorizer)

print("\n=== Test Email ===")
print(f"Email : {sample_email}")
print(f"Risk score : {score}")
print(f"Reasons : {reasons}")

/home/fatima-azzahra/2026-main/TP/TP1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Classification Report ===

              precision    recall  f1-score   support

       legit       0.33      0.32      0.32      3320
    phishing       0.33      0.33      0.33      3342
        spam       0.34      0.35      0.35      3338

    accuracy                           0.33     10000
   macro avg       0.33      0.33      0.33     10000
weighted avg       0.33      0.33      0.33     10000

=== Matrice de Confusion ===

[[1068 1122 1130]
 [1126 1094 1122]
 [1070 1100 1168]]

Modèle et vectorizer sauvegardés dans le dossier 'models/'

=== Test Email ===
Email : Urgent: verify your account now by clicking this link
Risk score : 0.30450611589109977
Reasons : {'length': np.int64(53), 'num_words': np.int64(9), 'num_exclam': np.int64(0), 'num_links': np.int64(0), 'num_digits': np.int64(0)}
